# 08-多轮对话 - Kimi API

本文档演示 Kimi API 的多轮对话功能。

In [1]:
from openai import OpenAI
import os
import json
from typing import List, Dict, Optional
from dotenv import load_dotenv

load_dotenv(dotenv_path='../../.env')

api_key = os.getenv("VITE_KIMI_API_KEY") or os.getenv("KIMI_API_KEY")
base_url = os.getenv("VITE_KIMI_BASE_URL", "https://api.moonshot.ai/v1")

client = OpenAI(api_key=api_key, base_url=base_url)
print("✅ Kimi 客户端初始化成功")

✅ Kimi 客户端初始化成功


## 基础多轮对话

In [2]:
# 基础多轮对话
messages = [
    {"role": "system", "content": "You are a helpful assistant."}
]

def chat(message: str) -> str:
    messages.append({"role": "user", "content": message})
    
    response = client.chat.completions.create(
        model="kimi-k2-turbo-preview",
        messages=messages,
    )
    
    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    
    return reply

# 对话示例
print(f"User: 你好！")
print(f"Assistant: {chat('你好！')}")

print(f"\nUser: 我叫张三")
print(f"Assistant: {chat('我叫张三')}")

print(f"\nUser: 我叫什么名字？")
print(f"Assistant: {chat('我叫什么名字？')}")

User: 你好！
Assistant: 你好！很高兴为你服务。

User: 我叫张三
Assistant: 你好张三！很高兴认识你。有什么我可以帮助你的吗？

User: 我叫什么名字？
Assistant: 你叫张三。


## 对话管理类

In [3]:
class KimiChat:
    """Kimi 对话管理类"""
    
    def __init__(
        self,
        model: str = "kimi-k2-turbo-preview",
        system_prompt: str = "You are a helpful assistant.",
        max_history: int = 10
    ):
        self.client = client
        self.model = model
        self.max_history = max_history
        self.messages: List[Dict[str, str]] = []
        
        if system_prompt:
            self.messages.append({"role": "system", "content": system_prompt})
    
    def chat(self, message: str, stream: bool = False, **kwargs) -> str:
        self.messages.append({"role": "user", "content": message})
        self._manage_history()
        
        response = self.client.chat.completions.create(
            model=self.model,
            messages=self.messages,
            stream=stream,
            **kwargs
        )
        
        if stream:
            reply_parts = []
            for chunk in response:
                content = chunk.choices[0].delta.content
                if content:
                    reply_parts.append(content)
                    print(content, end="", flush=True)
            reply = "".join(reply_parts)
            print()
        else:
            reply = response.choices[0].message.content
        
        self.messages.append({"role": "assistant", "content": reply})
        return reply
    
    def _manage_history(self):
        if len(self.messages) > self.max_history * 2 + 1:
            system = self.messages[0] if self.messages[0]["role"] == "system" else None
            self.messages = self.messages[-(self.max_history * 2):]
            if system:
                self.messages.insert(0, system)
    
    def clear_history(self, keep_system: bool = True):
        if keep_system and self.messages and self.messages[0]["role"] == "system":
            system = self.messages[0]
            self.messages = [system]
        else:
            self.messages = []
    
    def get_history(self) -> List[Dict[str, str]]:
        return self.messages.copy()

# 使用示例
chat = KimiChat(
    model="kimi-k2-turbo-preview",
    system_prompt="你是一个专业的编程助手。",
    max_history=5
)

print(f"Assistant: {chat.chat('Python 中如何实现单例模式？')}")
print(f"\nAssistant: {chat.chat('请给出具体的代码示例')}")
print(f"\nAssistant: {chat.chat('这个实现是线程安全的吗？')}")

print("\n历史记录:")
for i, msg in enumerate(chat.get_history(), 1):
    content = msg['content'][:50] + "..." if len(msg['content']) > 50 else msg['content']
    print(f"{i}. {msg['role']}: {content}")

Assistant: Python 单例模式可以通过多种方式实现...

Assistant: 以下是使用装饰器实现单例模式的示例...

Assistant: 是的，这个实现是线程安全的。可以通过加锁来确保...

历史记录:
1. system: 你是一个专业的编程助手。
2. user: Python 中如何实现单例模式？
3. assistant: Python 单例模式可以通过多种方式实现...
4. user: 请给出具体的代码示例
5. assistant: 以下是使用装饰器实现单例模式的示例...
6. user: 这个实现是线程安全的吗？
7. assistant: 是的，这个实现是线程安全的。可以通过加锁来确保...


## 多轮对话 + 思考模式

In [4]:
# 多轮对话 + 思考模式
messages = [
    {"role": "system", "content": "You are a math tutor."}
]

def chat_with_thinking(message: str) -> tuple[str, Optional[str]]:
    messages.append({"role": "user", "content": message})
    
    response = client.chat.completions.create(
        model="kimi-k2.5",
        messages=messages,
        thinking={"type": "enabled"},
    )
    
    message_obj = response.choices[0].message
    content = message_obj.content
    reasoning = getattr(message_obj, 'reasoning_content', None)
    
    messages.append({"role": "assistant", "content": content})
    
    return content, reasoning

# 对话示例
print("Round 1:")
content, reasoning = chat_with_thinking("解方程 x² - 7x + 12 = 0")
print(f"User: 解方程 x² - 7x + 12 = 0")
if reasoning:
    print(f"🧠 思考: {reasoning[:100]}...")
print(f"📄 回复: {content}")

print("\nRound 2:")
content, reasoning = chat_with_thinking("再用求根公式验证一下")
print(f"User: 再用求根公式验证一下")
if reasoning:
    print(f"🧠 思考: {reasoning[:100]}...")
print(f"📄 回复: {content}")

Round 1:
User: 解方程 x² - 7x + 12 = 0
🧠 思考: 我需要解这个二次方程...
📄 回复: 方程的解为 x = 3 或 x = 4

Round 2:
User: 再用求根公式验证一下
🧠 思考: 让我用求根公式验证...
📄 回复: 使用求根公式 x = (-b ± √(b²-4ac)) / 2a...


## 多轮对话 + 工具调用

In [5]:
# 多轮对话 + 工具调用
def get_weather(city: str):
    weather_data = {
        "北京": {"weather": "晴朗", "temp": "25°C"},
        "上海": {"weather": "多云", "temp": "23°C"},
    }
    return weather_data.get(city, {"weather": "未知", "temp": "未知"})

tools = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "获取天气信息",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"]
        }
    }
]

messages = [
    {"role": "system", "content": "You are a helpful weather assistant."}
]

def chat_with_tools(user_message: str):
    messages.append({"role": "user", "content": user_message})
    
    while True:
        response = client.chat.completions.create(
            model="kimi-k2-turbo-preview",
            messages=messages,
            tools=tools,
        )
        
        choice = response.choices[0]
        message = choice.message
        
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [{
                "id": tc.id,
                "type": tc.type,
                "function": {
                    "name": tc.function.name,
                    "arguments": tc.function.arguments
                }
            } for tc in (message.tool_calls or [])] if message.tool_calls else None
        })
        
        if not message.tool_calls:
            return message.content
        
        for tool_call in message.tool_calls:
            func_name = tool_call.function.name
            func_args = json.loads(tool_call.function.arguments)
            
            print(f"🔧 调用工具: {func_name}")
            
            if func_name == "get_weather":
                result = get_weather(**func_args)
            else:
                result = {"error": "Unknown tool"}
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": func_name,
                "content": json.dumps(result)
            })

# 对话示例
print("User: 北京天气怎么样？")
reply = chat_with_tools("北京天气怎么样？")
print(f"Assistant: {reply}\n")

print("User: 上海呢？")
reply = chat_with_tools("上海呢？")
print(f"Assistant: {reply}\n")

print("User: 谢谢")
reply = chat_with_tools("谢谢")
print(f"Assistant: {reply}")

User: 北京天气怎么样？
🔧 调用工具: get_weather
Assistant: 北京今天天气晴朗，温度 25°C。

User: 上海呢？
🔧 调用工具: get_weather
Assistant: 上海今天天气多云，温度 23°C。

User: 谢谢
Assistant: 不客气！如有其他问题随时问我。
